In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=40eE5NjSnF8pkGiGLOglrF9zGnp4Os&access_type=offline&code_challenge=kIdt56S79WvhNTHlfVJ0e_bvuoI-MYn_SmfYVqA5Zb0&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

hail_dir = os.path.dirname(hl.__file__)
session= GentropySession(hail_home=hail_dir)


Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/28 13:02:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
path_to_release_folder="gs://open-targets-data-releases/25.06/"


si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

In [3]:
ss1=SummaryStatistics.from_parquet(session,"gs://gwas_catalog_inputs/harmonised_summary_statistics/GCST90012877")

In [4]:
ss1.df.count()

10606767

In [5]:
from gentropy.common.genomic_region import GenomicRegion, KnownGenomicRegions
from gentropy.method.locus_breaker_clumping import LocusBreakerClumping

In [6]:
lbc = ss1.locus_breaker_clumping(
    baseline_pvalue_cutoff=1e-5,
    distance_cutoff=250_000,
    pvalue_cutoff=1e-8,
    flanking_distance=100_000,
)
wbc = ss1.window_based_clumping(distance=500_000, gwas_significance=1e-8)

clumped_result = LocusBreakerClumping.process_locus_breaker_output(
    lbc,
    wbc,
    large_loci_size=1_500_000,
)

clumped_result = clumped_result.exclude_region(
    GenomicRegion.from_known_genomic_region(KnownGenomicRegions.MHC),
    exclude_overlap=True,
)


clumped_result = clumped_result.annotate_locus_statistics_boundaries(
    ss1
)

In [7]:
sl_df=clumped_result.df.cache()

In [8]:
sl_df.count()

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/sql/pandas/functions.py:407: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.


25

In [9]:
sl_df.show()

+--------------------+------------+----------------+----------+---------+----------------+----------+--------------+--------------+-------------------------------+----------------+---------------+----------+---------+--------------------+
|        studyLocusId|     studyId|       variantId|chromosome| position|            beta|sampleSize|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|   standardError|qualityControls|locusStart| locusEnd|               locus|
+--------------------+------------+----------------+----------+---------+----------------+----------+--------------+--------------+-------------------------------+----------------+---------------+----------+---------+--------------------+
|c3b76f87277bcee69...|GCST90012877| 1_207577223_T_C|         1|207577223| -0.122752564739|      NULL|         1.403|           -23|                       0.177182| 0.0122652043685|           NULL| 207072474|207826726|[{1_207076447_T_C...|
|97694450999ebe9e3...|GCST90012877|11_121564

In [6]:
import gentropy.susie_finemapper as sfm

In [7]:
ld_matrix_paths={
    "pan_ukbb_bm_path": "gs://panukbb-ld-matrixes/UKBB.{POP}.ldadj",
    "ukbb_annotation_path": "gs://panukbb-ld-matrixes/UKBB.{POP}.aligned.parquet",
    "ld_matrix_template": "gs://gcp-public-data--gnomad/release/2.1.1/ld/gnomad.genomes.r2.1.1.{POP}.common.adj.ld.bm",
    "ld_index_raw_template": "gs://gcp-public-data--gnomad/release/2.1.1/ld/gnomad.genomes.r2.1.1.{POP}.common.ld.variant_indices.ht",
    "liftover_ht_path": "gs://gcp-public-data--gnomad/release/2.1.1/liftover_grch38/ht/genomes/gnomad.genomes.r2.1.1.sites.liftover_grch38.ht",
    "grch37_to_grch38_chain_path": "gs://hail-common/references/grch37_to_grch38.over.chain.gz",
}

In [17]:
tmp2 = StudyLocus(
    _df=session.spark.createDataFrame([sl_df.first()], sl_df.schema),
    _schema=StudyLocus.get_schema()
)

In [3]:
tmp1=StudyLocus.from_parquet(session, "gs://gwas_catalog_sumstats_susie/study_locus_lb_clumped/studyLocusId=00002a7ba2b9d1569ed0163a6426c3bd/")

In [19]:
study_locus = (
    session.spark.createDataFrame([sl_df.first()], sl_df.schema)
    .collect()[0]
)

In [20]:
import pandas as pd
pd.DataFrame.iteritems = pd.DataFrame.items

In [4]:
tmp1.df=tmp1.df.withColumn("studyId",f.lit("GCST90012877"))
tmp1.df=tmp1.df.withColumn("studyLocusId",f.lit("GCST90012877"))

In [62]:
from py4j.java_gateway import java_import
java_import(session.spark._sc._jvm, "org.apache.spark.sql.api.python.*")

In [63]:
# Check if Hail is already initialized
try:
    # Try to access Hail context
    hl.current_backend()
    print("Hail is already initialized")
except Exception as e:
    print(f"Hail not initialized: {e}")
    # Try to initialize
    hl.init(sc=session.spark.sparkContext)

Initializing Hail with default parameters...


Hail not initialized: 'JavaPackage' object is not callable


pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jar

TypeError: 'JavaPackage' object is not callable

In [8]:
x=sfm.SusieFineMapperStep.susie_finemapper_one_sl_row_gathered_boundaries(
    session=session,
    study_locus_row=tmp1.df.first(),
    study_index=si,
    ld_matrix_paths=ld_matrix_paths,
)

25/10/28 13:03:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
Initializing Hail with default parameters...                                    


TypeError: 'JavaPackage' object is not callable

In [49]:
tmp1.df.first()

Row(studyLocusId=None, studyType=None, variantId='16_15056648_C_A', chromosome='16', position=15056648, region=None, studyId='GCST90012877', beta=-0.0193, zScore=None, pValueMantissa=1.0959999561309814, pValueExponent=-18, effectAlleleFrequencyFromSource=0.29350000619888306, standardError=0.0022, subStudyDescription=None, qualityControls=None, finemappingMethod=None, credibleSetIndex=None, credibleSetlog10BF=None, purityMeanR2=None, purityMinR2=None, locusStart=14745896, locusEnd=15845083, sampleSize=None, ldSet=None, locus=[Row(is95CredibleSet=None, is99CredibleSet=None, logBF=None, posteriorProbability=None, variantId='16_15322866_C_G', pValueMantissa=8.128000259399414, pValueExponent=-1, beta=-0.0015, standardError=0.0066, r2Overall=None), Row(is95CredibleSet=None, is99CredibleSet=None, logBF=None, posteriorProbability=None, variantId='16_14992904_C_T', pValueMantissa=3.0199999809265137, pValueExponent=-1, beta=-0.007, standardError=0.0068, r2Overall=None), Row(is95CredibleSet=None,

In [26]:
study_locus_row=StudyLocus(
    _df=session.spark.createDataFrame([sl_df.first()], sl_df.schema),
    _schema=StudyLocus.get_schema()
).df.collect()[0]
gwas_df = session.spark.createDataFrame(
    [study_locus_row], StudyLocus.get_schema()
)

PySparkValueError: [LENGTH_SHOULD_BE_THE_SAME] obj and fields should be of the same length, got 15 and 27.

In [25]:
# Just use the row as-is without trying to force it into StudyLocus schema
study_locus_row = sl_df.first()
gwas_df = session.spark.createDataFrame([study_locus_row], sl_df.schema)

In [29]:
# Use the existing StudyLocus data which already has the correct schema
study_locus_row = sl.df.first()
gwas_df = session.spark.createDataFrame([study_locus_row], StudyLocus.get_schema())

In [37]:
sl_df.write.mode("overwrite").parquet("gs://genetics-portal-dev-analysis/yt4/for_FM_playaground.parquet")

In [38]:
sl_df1=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/for_FM_playaground.parquet")

In [43]:
# Get columns from both DataFrames
sl_df_cols = set(sl_df.columns)
sl_cols = set(sl_df1.columns)

print("Columns in sl_df:", len(sl_df_cols))
print(sorted(sl_df_cols))
print("\nColumns in sl.df:", len(sl_cols))
print(sorted(sl_cols))

print("\nColumns in sl_df but NOT in sl.df:")
print(sorted(sl_df_cols - sl_cols))

print("\nColumns in sl.df but NOT in sl_df:")
print(sorted(sl_cols - sl_df_cols))

print("\nCommon columns:")
print(sorted(sl_df_cols & sl_cols))

Columns in sl_df: 15
['beta', 'chromosome', 'effectAlleleFrequencyFromSource', 'locus', 'locusEnd', 'locusStart', 'pValueExponent', 'pValueMantissa', 'position', 'qualityControls', 'sampleSize', 'standardError', 'studyId', 'studyLocusId', 'variantId']

Columns in sl.df: 15
['beta', 'chromosome', 'effectAlleleFrequencyFromSource', 'locus', 'locusEnd', 'locusStart', 'pValueExponent', 'pValueMantissa', 'position', 'qualityControls', 'sampleSize', 'standardError', 'studyId', 'studyLocusId', 'variantId']

Columns in sl_df but NOT in sl.df:
[]

Columns in sl.df but NOT in sl_df:
[]

Common columns:
['beta', 'chromosome', 'effectAlleleFrequencyFromSource', 'locus', 'locusEnd', 'locusStart', 'pValueExponent', 'pValueMantissa', 'position', 'qualityControls', 'sampleSize', 'standardError', 'studyId', 'studyLocusId', 'variantId']


In [33]:
StudyLocus.get_schema()

StructType([StructField('studyLocusId', StringType(), False), StructField('studyType', StringType(), True), StructField('variantId', StringType(), False), StructField('chromosome', StringType(), True), StructField('position', IntegerType(), True), StructField('region', StringType(), True), StructField('studyId', StringType(), False), StructField('beta', DoubleType(), True), StructField('zScore', DoubleType(), True), StructField('pValueMantissa', FloatType(), True), StructField('pValueExponent', IntegerType(), True), StructField('effectAlleleFrequencyFromSource', FloatType(), True), StructField('standardError', DoubleType(), True), StructField('subStudyDescription', StringType(), True), StructField('qualityControls', ArrayType(StringType(), False), True), StructField('finemappingMethod', StringType(), True), StructField('credibleSetIndex', IntegerType(), True), StructField('credibleSetlog10BF', DoubleType(), True), StructField('purityMeanR2', DoubleType(), True), StructField('purityMinR

In [42]:
x=sfm.SusieFineMapperStep.susie_finemapper_one_sl_row_gathered_boundaries(
    session=session,
    study_locus_row=sl_df1.first(),
    study_index=si,
    ld_matrix_paths=ld_matrix_paths,
)

PySparkValueError: [LENGTH_SHOULD_BE_THE_SAME] obj and fields should be of the same length, got 15 and 27.

In [44]:
sl_df1.select("studyId").show(1)

+------------+
|     studyId|
+------------+
|GCST90012877|
+------------+
only showing top 1 row



In [9]:
from gentropy.common.session import Session
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.method.susie_inf import SUSIE_inf
from gentropy.susie_finemapper import SusieFineMapperStep

In [12]:
import pandas as pd
ld = np.loadtxt("../tests/gentropy/data_samples/01_test_ld.csv", delimiter=",")
z = pd.read_csv("../tests/gentropy/data_samples/01_test_z.csv")
z = np.array(z.iloc[:, 1])
lbf_moments = np.loadtxt("../tests/gentropy/data_samples/01_test_lbf_moments.csv")
lbf_mle = np.loadtxt("../tests/gentropy/data_samples/01_test_lbf_mle.csv")
sample_data_for_susie_inf=[ld, z, lbf_moments, lbf_mle]

In [14]:
sample_summary_statistics=SummaryStatistics(
        _df=session.spark.read.parquet("../tests/gentropy/data_samples/sumstats_sample"),
        _schema=SummaryStatistics.get_schema(),
    )

In [15]:
ld = sample_data_for_susie_inf[0]
z = sample_data_for_susie_inf[1]
susie_output = SUSIE_inf.susie_inf(
    z=z,
    LD=ld,
    est_tausq=False,
)
gwas_df = sample_summary_statistics._df.withColumn(
    "z", f.col("beta") / f.col("standardError")
).filter(f.col("z").isNotNull())
gwas_df = gwas_df.limit(21)

In [16]:
L1 = SusieFineMapperStep.susie_inf_to_studylocus(
    susie_output=susie_output,
    session=session,
    studyId="sample_id",
    region="sample_region",
    variant_index=gwas_df,
    cs_lbf_thr=2,
    ld_matrix=ld,
    lead_pval_threshold=1,
    purity_mean_r2_threshold=0,
    purity_min_r2_threshold=0,
    sum_pips=0.99,
    ld_min_r2=1,
    locusStart=1,
    locusEnd=2,
)

25/10/28 15:34:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:34:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:34:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:34:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:34:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:34:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 1

In [18]:
L1.df.show(truncate=False)

25/10/28 15:35:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:35:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:35:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:35:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:35:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 15:35:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/28 1

+----------------+--------------------------------+---------+-------------+--------------------------------------------------------------------------+-------------+----------+--------+-----------------+------------------+------------+-----------+--------------------+--------------+--------------+----------+--------+-------------------+
|credibleSetIndex|studyLocusId                    |studyId  |region       |locus                                                                     |variantId    |chromosome|position|finemappingMethod|credibleSetlog10BF|purityMeanR2|purityMinR2|zScore              |pValueMantissa|pValueExponent|locusStart|locusEnd|beta               |
+----------------+--------------------------------+---------+-------------+--------------------------------------------------------------------------+-------------+----------+--------+-----------------+------------------+------------+-----------+--------------------+--------------+--------------+----------+--------+-------

In [20]:
gwas_df.show(truncate=False)

+----------+-------------+----------+--------+--------------+--------------+---------------------+--------------------+-------------------------------+--------------------+
|studyId   |variantId    |chromosome|position|pValueMantissa|pValueExponent|beta                 |standardError       |effectAlleleFrequencyFromSource|z                   |
+----------+-------------+----------+--------+--------------+--------------+---------------------+--------------------+-------------------------------+--------------------+
|GCST005523|18_184476_G_A|18        |184476  |6.863         |-1            |-0.011364330079049759|0.02813779634751929 |NULL                           |-0.4038813110555358 |
|GCST005523|18_199462_A_G|18        |199462  |5.885         |-1            |0.016857117066422806 |0.03115854766090391 |NULL                           |0.5410110012147396  |
|GCST005523|18_213151_A_C|18        |213151  |7.436         |-1            |-0.00904074465214907 |0.027639950507337172|NULL            